In [1]:
import pandas as pd
import requests

from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

In [2]:
from selenium.webdriver.chrome.options import Options

URL = "https://jpdb.io/deck?id=global"

options = Options()
options.binary_location = r"C:\Users\firas\AppData\Local\BraveSoftware\Brave-Browser\Application\brave.exe"

# Launch a Chrome-based browser window
driver = webdriver.Chrome(options=options)

# Go the URL website
driver.get(URL)

# Close the window
# driver.quit()

In [81]:

driver.current_url

'https://jpdb.io/deck?id=global&offset=100#a'

In [4]:
driver.title

'Log in or Sign up – jpdb'

In [5]:
driver.page_source

'<html class="dark-mode"><head><meta http-equiv="Content-type" content="text/html; charset=utf-8"><meta http-equiv="Content-language" content="en"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0"><title>Log in or Sign up – jpdb</title><link rel="stylesheet" media="screen" href="/static/be945648d268.css"><link rel="stylesheet" media="screen" href="/static/32a5a4227a9c.css"><link rel="apple-touch-icon" sizes="180x180" href="/static/533228467534.png"><link rel="icon" type="image/png" sizes="32x32" href="/static/414de0c1e6b5.png"><link rel="icon" type="image/png" sizes="16x16" href="/static/8f8d7f6ca822.png"><link rel="manifest" href="/static/9919db124702.webmanifest"><link rel="search" type="application/opensearchdescription+xml" title="jpdb" href="/static/opensearch.xml"><script defer="" type="text/javascript" src="/static/0c282812caef.js"></script><script>function oneshot(e, n, f) { var g = func

In [6]:
driver.refresh()

In [78]:
master = pd.DataFrame({
    'keyword':[],
    'status':[]
})

In [94]:
cards = {'keyword':[], 'status':[]}

vocab_list = driver.find_element(By.CLASS_NAME, "vocabulary-list")

# ALL CARDS
for card in vocab_list.find_elements(By.CSS_SELECTOR, ":scope > div"):

    # GENERAL DIV
    general_div = card.find_element(By.TAG_NAME, "div")

    # VOCAB SPELLING
    vocab_spelling = general_div.find_element(By.TAG_NAME, "div")

    # A TAG for vocab
    a_tag = vocab_spelling.find_element(By.TAG_NAME, "a")

    # DIV for known status
    known_div = vocab_spelling.find_element(By.TAG_NAME, "div")

    # SPECIFIC DIV for known status
    known_status = known_div.find_element(By.TAG_NAME, "div").text


    # GET KEYWORD

    # RUBY TAGS
    rubies = a_tag.find_elements(By.TAG_NAME, "ruby")

    # Build the keyword
    characters = []
    
    # For each ruby tag
    for ruby in rubies:

        # If the character is not a kanji hence has no furigana, leave it as is and add to the list
        if len(ruby.find_elements(By.TAG_NAME, "rt")) == 0:
            characters.append(ruby.text)

        # Otherwise, if it is a kanji, remove the furigana that follows, clean the string and add it to the list
        else:
            furigana = ruby.find_element(By.TAG_NAME, "rt").text

            characters.append(ruby.text.replace(furigana,"").strip())
    
    # Build the keyword from the characters
    keyword = "".join(characters)

    # Add keyword and known status to the dictionary
    cards['keyword'].append(keyword)
    cards['status'].append(known_status)

# Convert cards dictionary to Pandas dataframe
cards = pd.DataFrame(cards)

master = pd.concat([master,cards],axis='rows')

cards


,keyword,status
0,の,Never forget
1,に,Never forget
2,よ,Never forget
3,が,Never forget
4,を,Never forget
5,も,Never forget
6,んだ,Never forget
7,は,Never forget
8,で,Never forget
9,こと,Never forget


In [95]:
master

,keyword,status
0,時,Locked
1,信じる,Never forget
2,けど,Never forget
3,小さな,Never forget
4,泣く,Locked
...,...,...
45,言う,Never forget
46,くれる,Never forget
47,だから,Never forget
48,ん,Never forget


In [96]:
master.to_csv(r'JPDB_all_vocabulary_knowledge_scraped.csv',index=False)